In [2]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

# তোমার পাথ
path = '/content/drive/MyDrive/Thesis file/Feature_Extract_Formants/'

extractors = ['AlexNet', 'CustomCNN', 'DenseNet121', 'EfficientNetB0', 'InceptionV3',
              'MobileNetV2', 'ResNet50', 'VGG16', 'ViT', 'Xception']

classifiers = {
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

results = pd.DataFrame(columns=['Feature Extractor'] + list(classifiers.keys()) + ['Best', 'Best Accuracy'])
results['Feature Extractor'] = extractors

for extractor in extractors:
    print(f'Processing {extractor} ...')

    f1_file = f'{path}F1_Features_{extractor}.csv'
    f2_file = f'{path}F2_Features_{extractor}.csv'

    df1 = pd.read_csv(f1_file)
    df2 = pd.read_csv(f2_file)

    # Step 1: লেবেল কলাম খুঁজে বের করা (সাধারণ নাম)
    possible_label_cols = ['label', 'class', 'target', 'phoneme', 'category', 'Label', 'Class', 'phoneme_label']
    label_col = None
    for col in possible_label_cols:
        if col in df1.columns:
            label_col = col
            break

    # Step 2: যদি কোনোটাই না পায় → ধরে নিচ্ছি দ্বিতীয় কলামটাই লেবেল (অধিকাংশ ফিচার ফাইলে এমনই হয়)
    if label_col is None:
        label_col = df1.columns[1]   # প্রথম কলাম সাধারণত index/filename, দ্বিতীয়টা লেবেল
        print(f"   → লেবেল কলাম ধরা হলো: {label_col}")

    # Step 3: filename, path, Unnamed ইত্যাদি বাদ দিয়ে ফিচার নেওয়া
    def clean_features(df):
        df = df.copy()
        # filename, path, Unnamed কলাম বাদ
        drop_cols = [c for c in df.columns if any(x in c.lower() for x in ['filename', 'path', 'unnamed', 'index'])]
        df.drop(columns=drop_cols, errors='ignore', inplace=True)
        return df

    df1_clean = clean_features(df1)
    df2_clean = clean_features(df2)

    # ফিচার আর লেবেল আলাদা করা
    X1 = df1_clean.drop(columns=[label_col], errors='ignore')
    X2 = df2_clean.drop(columns=[label_col], errors='ignore')
    y = df1_clean[label_col]   # লেবেল F1 থেকে নিচ্ছি

    # ফিচার জোড়া
    X = pd.concat([X1.reset_index(drop=True), X2.reset_index(drop=True)], axis=1)

    # Step 4: stratify সমস্যা এড়ানোর জন্য → যদি কোনো ক্লাসে ১টা স্যাম্পল থাকে, তাহলে stratify বন্ধ
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    except ValueError:
        print(f"   → {extractor}: কিছু ক্লাসে ১টা স্যাম্পল → stratify ছাড়া স্প্লিট করা হলো")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    best_acc = 0
    best_name = ''

    for name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = round(accuracy_score(y_test, y_pred), 4)

        results.loc[results['Feature Extractor'] == extractor, name] = acc

        if acc > best_acc:
            best_acc = acc
            best_name = name

    results.loc[results['Feature Extractor'] == extractor, 'Best'] = best_name
    results.loc[results['Feature Extractor'] == extractor, 'Best Accuracy'] = best_acc

# সেভ
output_file = path + 'Full_Summary_Table.xlsx'
results.to_excel(output_file, index=False)
print('\nসব শেষ! তোমার এক্সেল তৈরি হয়ে গেছে!')
print(output_file)
display(results)

Mounted at /content/drive
Processing AlexNet ...
Processing CustomCNN ...
Processing DenseNet121 ...
Processing EfficientNetB0 ...
Processing InceptionV3 ...
Processing MobileNetV2 ...
Processing ResNet50 ...
Processing VGG16 ...
Processing ViT ...
Processing Xception ...

সব শেষ! তোমার এক্সেল তৈরি হয়ে গেছে!
/content/drive/MyDrive/Thesis file/Feature_Extract_Formants/Full_Summary_Table.xlsx


,Feature Extractor,AdaBoost,Decision Tree,Extra Trees,Gradient Boosting,KNN,Logistic Regression,Naive Bayes,Random Forest,SVM,Best,Best Accuracy
0,AlexNet,0.9048,0.7619,0.8857,0.9048,0.8952,0.9048,0.7619,0.8667,0.8952,AdaBoost,0.9048
1,CustomCNN,0.8095,0.7143,0.8857,0.819,0.8286,0.8476,0.6571,0.8476,0.8571,Extra Trees,0.8857
2,DenseNet121,0.8762,0.8095,0.8762,0.8857,0.8667,0.8857,0.819,0.8762,0.8952,SVM,0.8952
3,EfficientNetB0,0.9524,0.8857,0.8952,0.9048,0.8286,0.9429,0.8571,0.9048,0.8476,AdaBoost,0.9524
4,InceptionV3,0.9048,0.6952,0.8476,0.8381,0.8381,0.9048,0.819,0.8286,0.8667,AdaBoost,0.9048
5,MobileNetV2,0.9048,0.8476,0.8667,0.8762,0.8952,0.9238,0.8286,0.8952,0.8952,Logistic Regression,0.9238
6,ResNet50,0.8571,0.8286,0.8667,0.9143,0.8667,0.9048,0.8762,0.8667,0.8857,Gradient Boosting,0.9143
7,VGG16,0.8857,0.8286,0.8762,0.8762,0.8571,0.8857,0.7905,0.8762,0.8857,AdaBoost,0.8857
8,ViT,0.8667,0.8381,0.8095,0.8571,0.7905,0.8667,0.819,0.8,0.8476,AdaBoost,0.8667
9,Xception,0.8952,0.8286,0.8571,0.8667,0.7714,0.9143,0.8381,0.8762,0.8667,Logistic Regression,0.9143
